# OBR Fase 3 — Segmentação neural da linha

Treina LinhaNet e LR-ASPP somente em treino/validação. O teste de 591 imagens não faz parte do pacote. Selecione uma GPU T4 em **Ambiente de execução → Alterar tipo de ambiente de execução** antes de começar.

In [ ]:
!git clone -q https://github.com/DaviBonetto/OBR.git /content/OBR
%cd /content/OBR
!pip install -q -e ".[treinamento]"

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

PACOTE = Path('/content/drive/MyDrive/OBR/fase3_dataset_inicial.zip')
RESULTADOS = Path('/content/drive/MyDrive/OBR/resultados_fase3')
assert PACOTE.is_file(), f'Copie o pacote para: {PACOTE}'
RESULTADOS.mkdir(parents=True, exist_ok=True)

In [ ]:
import hashlib
import shutil

HASH_ESPERADO = '244b4f7b5d15e495fa9059cd8b19f1bb56496c01bb42f6e506bccb7e53bf2c9d'
hash_obtido = hashlib.sha256(PACOTE.read_bytes()).hexdigest()
assert hash_obtido == HASH_ESPERADO, (hash_obtido, HASH_ESPERADO)
DATASET = Path('/content/fase3_dataset_inicial')
if DATASET.exists():
    shutil.rmtree(DATASET)
shutil.unpack_archive(PACOTE, DATASET)
print('Dataset verificado e extraído:', DATASET)

In [ ]:
import torch

assert torch.cuda.is_available(), 'Ative a GPU T4 antes do treinamento completo.'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Experimento A — LinhaNet
32 mil parâmetros; candidata principal de baixa latência.

In [ ]:
import subprocess

subprocess.run([
    'obr-treinar-segmentacao', '--dataset', str(DATASET),
    '--saida', str(RESULTADOS / 'linhanet_v1'),
    '--arquitetura', 'linhanet',
], check=True)

## Experimento B — LR-ASPP MobileNetV3
Modelo de maior capacidade para comparação e active learning.

In [ ]:
subprocess.run([
    'obr-treinar-segmentacao', '--dataset', str(DATASET),
    '--saida', str(RESULTADOS / 'lraspp_v1'),
    '--arquitetura', 'lraspp_mobilenet_v3_large',
], check=True)

In [ ]:
import json

for experimento in ('linhanet_v1', 'lraspp_v1'):
    caminho = RESULTADOS / experimento / 'manifesto.json'
    print(experimento, json.loads(caminho.read_text()))
print('Não abra o teste. Envie os dois manifestos e históricos para comparação.')